In [1]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime
from pathlib import Path

In [ ]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [2]:
#Example for reading in one file:
result_dir = Path('/glade/derecho/scratch/demurray/archive/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001/atm/hist')
file = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam.h2.2022-12-31-00000.nc'
file_to_open = result_dir / file
nc_load = xr.open_dataset(file_to_open, engine='netcdf4')
nc_load

<xarray.Dataset>
Dimensions:                  (lat: 192, lon: 288, lev: 32, ilev: 33, time: 1,
                              nbnd: 2)
Coordinates:
  * lat                      (lat) float64 -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                      (lon) float64 0.0 1.25 2.5 ... 356.2 357.5 358.8
  * lev                      (lev) float64 3.643 7.595 14.36 ... 976.3 992.6
  * ilev                     (ilev) float64 2.255 5.032 10.16 ... 985.1 1e+03
  * time                     (time) datetime64[ns] 2022-12-31
Dimensions without coordinates: nbnd
Data variables: (12/101)
    gw                       (lat) float64 ...
    hyam                     (lev) float64 ...
    hybm                     (lev) float64 ...
    P0                       float64 ...
    hyai                     (ilev) float64 ...
    hybi                     (ilev) float64 ...
    ...                       ...
    soa5_a2DDF               (time, lat, lon) float32 ...
    soa5_a2SFWET             (time, lat, lon) float32 ...
    soa5_c2DDF               (time, lat, lon) float32 ...
    soa5_c2SFWET             (time, lat, lon) float32 ...
    wet_deposition_NHx_as_N  (time, lat, lon) float32 ...
    wet_deposition_NOy_as_N  (time, lat, lon) float32 ...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001
    logname:           demurray
    host:              derecho8
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [ ]:
#extract grid variables
lat = nc_load['lat']
lon = nc_load['lon']
var_sel = nc_load['bc_a1SFWET']

# CAM-chem writes the month average at midnight - i.e. the start of the next month.
# Reconfigure the time variable
time = nc_load['time']
time2 = pd.to_datetime(time.values,format='%Y-%m-%dT%H:%M:%S')+DateOffset(months=-1,days=+14)
#print("file time: ", time[0].values, "---> converted time: ", time2[0])

In [ ]:
#Select and extract the location
name_select = "Boulder"
lat_select = 40.0150
lon_select = 360-105.2705 # model longitude is from 0 to 360

lat_i = find_index(lat.values, lat_select)
lon_i = find_index(lon.values, lon_select)

print(name_select, " latitude: ", lat_select, "---> nearest: ", lat[lat_i].values)
print(name_select, " longitude: ", lon_select, "---> nearest: ", lon[lon_i].values)

conversions = 2.628e6*10000 #(seconds to months and m2 to hectare)
var_srf = var_sel.isel(lat=lat_i,lon=lon_i)*conversions*-1 #Convert to positive values, negative because its a flux TO surface

In [ ]:
#plot over time at Boulder CO
plt.figure(figsize=(20,8))
plt.plot(time2, var_srf, label='CAM-chem bc_a1')
plt.title('bc_a1 wet deposition at ' + name_select)
plt.xlabel('time')
plt.ylabel('bc_a1 wet deposition (kg/ha/month)')
plt.legend()
plt.show()